# VINS pose vs FC EKF — direct comparison (rekon10 260705)

**coordinator [#42](https://github.com/symmatree/coordinator/issues/42).** We captured the estimator
*inputs* (`chobits_imu` + `chobits_features`) on the vehicle with `vio-ipc-record`, then replayed them
**offline through the real `vins_fusion`** to regenerate a pose trajectory (see
[`docs/vio-offline-replay.md`](../docs/vio-offline-replay.md)). This notebook aligns that regenerated
VINS pose to the FC dataflash log and asks the plain question: **how close is the VINS trajectory to the
EKF/GPS ground truth?**

The FC `.bin` is 1980-dated (no RTC at log open), so we align by **angular-speed cross-correlation**
(both the OAK-D and the FC gyro see the same rotation), not wall-clock. VINS lives in its own gravity-aligned
world frame; a similarity (Umeyama) fit absorbs the rotation/translation and — importantly — **reports the
scale**, which turns out to be the story.

Run it on either capture by changing the parameters cell:
- **handheld** (`260705-handheld-noarm`) — disarmed, hand-carried, motors off.
- **armed** (`260705-vio-logged`) — a real powered flight.

In [ ]:
# --- Parameters (papermill-injected) ---
data_dir     = "/home/jovyan/datasets/flights/rekon10/260705-vio-logged"
fc_bin       = data_dir + "/1980-01-06 10-45-23.bin"
pose_csv     = data_dir + "/wave-20260705-114708.vinspose.csv"
run_name     = "armed"
replay_speed = 0.9      # input_replayer --speed used to make pose_csv (rescales its replay clock)
vel_diverge  = 6.0      # m/s: VINS speed above this = it has run away (non-physical for this vehicle)
repo_analysis = "/home/jovyan/coordinator/analysis"


In [ ]:
%matplotlib inline
import sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, repo_analysis)
import vio_ekf_compare as vc
from ardupilot_log import parse_log
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
print("comparing:", run_name)
print("  pose:", pose_csv.split('/')[-1])
print("  fc  :", fc_bin.split('/')[-1])

## 1. The regenerated VINS pose, as-is

First just look at what `vins_fusion` produced from the replayed inputs — **before** any alignment.
The tell-tale of VIO health is whether position stays physically plausible.

In [ ]:
v = vc.load_vio_pose(pose_csv, replay_speed)
print(f"{len(v)} pose samples, {v.te.iloc[-1]:.0f}s capture-time")
print(f"position range (m):  x[{v.px.min():.0f},{v.px.max():.0f}]  "
      f"y[{v.py.min():.0f},{v.py.max():.0f}]  z[{v.pz.min():.0f},{v.pz.max():.0f}]")
print(f"speed |v| (m/s):     mean {v.speed.mean():.1f}   max {v.speed.max():.1f}")
print(f"max |position| = {np.abs(v[['px','py','pz']].values).max():.0f} m")

fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(v.te, v.px, label="pN"); ax[0].plot(v.te, v.py, label="pE"); ax[0].plot(v.te, v.pz, label="pD")
ax[0].set_ylabel("VINS position (m)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[0].set_title(f"{run_name}: raw VINS pose over the whole replay")
ax[1].semilogy(v.te, v.speed.clip(1e-3), color="purple"); ax[1].axhline(vel_diverge, color="red", ls="--", lw=.8)
ax[1].set_ylabel("VINS |v| (m/s, log)"); ax[1].set_xlabel("capture time (s)"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

Read this before scrolling on: position sits at ~0 while the vehicle is stationary on the ground,
holds through the first real motion, then **runs away to hundreds/thousands of metres** with absurd
speeds. So VINS tracks briefly, then diverges. The rest of the notebook pins down *when* it diverges and
*how well it tracked before that* — by comparing to the EKF.

## 2. Time-align VINS to the FC log

Slide the VINS body-angular-speed envelope over the FC `|gyro|` envelope (using only the pre-divergence
part, since the runaway corrupts the correlation). The peak gives the lag `fc_t ≈ vio_t + lag`; the peak
NCC is the alignment quality.

In [ ]:
fc = vc.load_fc(fc_bin)
dv = np.argmax(v.speed.values > vel_diverge) if (v.speed.values > vel_diverge).any() else len(v)-1
pre = v.iloc[:dv+1]
lag, ncc = vc.align_time(pre.te.values, pre.w.values, fc["imu0"].t_s.values, fc["imu0"].w.values)
v["tfc"] = v.te + lag
print(f"lag = {lag:.2f}s   alignment NCC = {ncc:.3f}   (FC log {fc['dur']:.0f}s)")

### Sanity-check the alignment against a *physical* event

If the lag is right, the moment VINS starts moving should coincide with the FC seeing motion — motors
spinning up / the EKF altitude climbing (armed), or the gyro waking up (handheld). Overlay them.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.2))
xkf = fc["xkf"]
ax.plot(xkf.t_s, -xkf.PD, color="crimson", lw=1, label="EKF altitude (-PD)")
ax.plot(v.tfc, v.speed.clip(0, 30), color="steelblue", lw=.7, alpha=.7, label="VINS |v| (clipped 30)")
rc = fc["F"].get("RCOU")
if rc is not None:
    a2 = ax.twinx(); a2.plot(rc.t_s, rc.C1, color="gray", lw=.6, alpha=.5); a2.set_ylabel("motor C1 (us)")
ax.set_xlim(lag-20, lag + v.te.iloc[-1] + 10)
ax.set_xlabel("FC t_s (s)"); ax.set_ylabel("alt (m) / VINS |v|"); ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=.3)
ax.set_title(f"{run_name}: VINS motion onset vs FC (motors / EKF climb) — visual alignment check")
plt.tight_layout(); plt.show()

## 3. The valid window, and VINS-vs-EKF inside it

Define the window from **motion onset** to **divergence** (VINS speed running away, or the FC recording an
aggressive maneuver — the hand barrel-roll / hard yaw that breaks tracking). Then Umeyama-fit the VINS
position to the EKF NED position over that window.

Watch the **scale** the fit recovers: VINS is metric (stereo baseline), so an honest track gives scale ≈ 1.

In [ ]:
t0, t_div, reason = vc.find_window(v, fc, lag, vel_diverge=vel_diverge)
win = v[(v.tfc >= t0) & (v.tfc <= t_div)].copy()
gt = np.c_[np.interp(win.tfc, xkf.t_s, xkf.PN),
           np.interp(win.tfc, xkf.t_s, xkf.PE),
           np.interp(win.tfc, xkf.t_s, xkf.PD)]
vio = win[["px","py","pz"]].values
R, c, t = vc.umeyama(vio, gt, with_scale=True)
vio_al = (c*(R@vio.T).T + t)
err = np.linalg.norm(vio_al - gt, axis=1)
# rigid fit too (scale forced to 1) — shows the raw metric disagreement
Rr, _, tr = vc.umeyama(vio, gt, with_scale=False)
vio_rigid = (Rr@vio.T).T + tr
err_rigid = np.linalg.norm(vio_rigid - gt, axis=1)
print(f"window: FC t_s [{t0:.1f}, {t_div:.1f}]  ({t_div-t0:.1f}s, {len(win)} samples)  cut: {reason}")
print(f"Umeyama SCALE = {c:.3f}   -> VINS reports ~{1/c:.1f}x the real motion")
print(f"ATE (scale absorbed): rmse {np.sqrt((err**2).mean()):.2f}  max {err.max():.2f} m")
print(f"error with scale forced to 1 (true metric gap): rmse {np.sqrt((err_rigid**2).mean()):.2f}  max {err_rigid.max():.2f} m")
print(f"EKF path {np.linalg.norm(np.diff(gt,axis=0),axis=1).sum():.1f} m   "
      f"VINS path {np.linalg.norm(np.diff(vio,axis=0),axis=1).sum():.1f} m")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
ax[0].plot(gt[:,1], gt[:,0], color="crimson", lw=1.6, label="EKF (PE,PN)")
ax[0].plot(vio_rigid[:,1], vio_rigid[:,0], color="steelblue", lw=1.4, label="VINS (rigid, scale=1)")
ax[0].plot(vio_al[:,1], vio_al[:,0], color="seagreen", lw=1, ls="--", label=f"VINS (scale={c:.2f})")
ax[0].scatter([gt[0,1]],[gt[0,0]], c="k", s=25, zorder=5)
ax[0].set_aspect("equal","datalim"); ax[0].set_xlabel("East (m)"); ax[0].set_ylabel("North (m)")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3); ax[0].set_title("horizontal path in the valid window")
ax[1].plot(win.tfc, err_rigid, color="steelblue", label="scale=1")
ax[1].plot(win.tfc, err, color="seagreen", ls="--", label=f"scale={c:.2f}")
ax[1].axhline(1, color="orange", ls=":", lw=.8, label="1 m")
ax[1].set_xlabel("FC t_s (s)"); ax[1].set_ylabel("|VINS - EKF| (m)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
ax[1].set_title("position error growth")
plt.tight_layout(); plt.show()

## 4. The idle period: is VINS at least *steady* when nothing moves?

Both logs start with the vehicle sitting on the concrete waiting for a GPS fix. So during idle we expect
**VINS ≈ perfectly still** and the **EKF/GPS wandering then settling** as the fix firms up. This is the one
regime where VINS should look *better* than GPS.

In [ ]:
idle = v[v.tfc < t0]
xki = xkf[xkf.t_s < t0]
fig, ax = plt.subplots(figsize=(12, 3))
if len(idle):
    ax.plot(idle.tfc, idle.px-idle.px.iloc[0], lw=.9, label="VINS N")
    ax.plot(idle.tfc, idle.py-idle.py.iloc[0], lw=.9, label="VINS E")
if len(xki):
    ax.plot(xki.t_s, xki.PN-xki.PN.iloc[0], color="crimson", lw=.9, alpha=.7, label="EKF N")
    ax.plot(xki.t_s, xki.PE-xki.PE.iloc[0], color="darkred", lw=.9, alpha=.7, label="EKF E")
ax.set_xlabel("FC t_s (s)"); ax.set_ylabel("position - start (m)"); ax.legend(fontsize=8); ax.grid(alpha=.3)
ax.set_title(f"{run_name}: idle-on-ground — VINS drift vs GPS/EKF settling")
plt.tight_layout(); plt.show()
if len(idle):
    print(f"VINS idle drift: {np.abs(idle[['px','py']].values - idle[['px','py']].values[0]).max():.2f} m over {idle.tfc.iloc[-1]-idle.tfc.iloc[0]:.0f}s")
if len(xki):
    print(f"EKF idle wander: {np.abs(xki[['PN','PE']].values - xki[['PN','PE']].values[0]).max():.2f} m (GPS settling)")

## 5. Summary

In [ ]:
m = vc.compare(pose_csv, fc_bin, run_name, replay_speed, vel_diverge=vel_diverge, make_plot=False)
m["umeyama_scale_x"] = round(1/m["umeyama_scale"], 1) if m["umeyama_scale"] else None
print(json.dumps(m, indent=2))

**What this run shows** (read together with the numbers above):

- VINS **initialises and holds** while the vehicle is stationary, and is *steadier than GPS* during the
  idle-on-ground period (§4) — the input path, timing, and estimator are all working.
- Once real motion begins, VINS **over-reports the motion by a large scale factor** (§3, `umeyama_scale`)
  and then **diverges** within seconds. The `usable_track_s` figure is how long it stayed within 1 m of the
  EKF after moving.
- So the limiting factor is **not** the harness or the replay — it's VINS metric accuracy on this rig with
  the current *seed* calibration (identity cam↔IMU extrinsics, un-refined IMU noise / time offset). That is
  the open VIO-quality work: a real cam↔IMU calibration and scale check before VINS pose is trustworthy.

The companion notebook `vio-input-alignment.ipynb` shows the *input-side* evidence (incl. that the OAK-D IMU
sees the motor vibration ~400–500× in the armed run); this notebook shows the *output-side* consequence.